# CREAM Progressive Graph Noise

Loads progressive graph-noise runs and plots task accuracy, concept accuracy, CCI, PFI, and intervention curves as a function of graph distance.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'experiments').exists() and (PROJECT_ROOT.parent / 'experiments').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

dataset_roots = {
    'cub': PROJECT_ROOT / 'experiments/CUB/train_cbm/Standard_CUB/cub_progressive_noise/CREAM_cub_progressive_noise',
    'celeba': PROJECT_ROOT / 'experiments/CelebA/train_cbm/Standard_CelebA/celeba_progressive_noise/CREAM_celeba_progressive_noise',
}
metadata_paths = {
    'cub': PROJECT_ROOT / 'data/CUB/progressive_noise_graph/progressive_noise_metadata.csv',
    'celeba': PROJECT_ROOT / 'data/CelebA/progressive_noise_graph/progressive_noise_metadata.csv',
}
baseline_roots = {
    'cub': PROJECT_ROOT / 'experiments/CUB/train_cbm/Standard_CUB/CREAM/CREAM_best',
}
summary_out = PROJECT_ROOT / 'notebook/progressive_noise_summary_values.csv'
intervention_out = PROJECT_ROOT / 'notebook/progressive_noise_intervention_values.csv'

BASELINE_VARIANT = 'original_CREAM_best'
METRIC_COLUMNS = {
    'test_task_accuracy',
    'test_concept_accuracy',
    'test_dropout_task_accuracy',
    'CCI',
    'PFI_concept_importance',
    'PFI_side_importance',
}
SKIP_BASELINE_CSV_TOKENS = (
    'intervention',
    'concept_activation',
    'exogenous',
    'C_and_S',
    'train_set',
    'test_set',
    'perc_',
)

def available(columns, requested):
    return [col for col in requested if col is not None and col in columns]

def _read_first_metric_row(csv_path):
    try:
        df = pd.read_csv(csv_path)
    except Exception as exc:
        print(f'Skipping unreadable CSV {csv_path}: {exc}')
        return None
    if df.empty or not (METRIC_COLUMNS & set(df.columns)):
        return None
    return df.iloc[0].to_dict()

def _dedupe_paths(paths):
    seen = set()
    out = []
    for path in paths:
        key = path.resolve() if path.exists() else path
        if key not in seen:
            out.append(path)
            seen.add(key)
    return out

def load_last_metrics(dataset_roots, metadata_paths):
    rows = []
    for dataset, root in dataset_roots.items():
        for csv_path in sorted(root.glob('noise_*/last_metrics/*.csv')):
            row = _read_first_metric_row(csv_path)
            if row is None:
                continue
            row['dataset'] = dataset
            row['noise_percent'] = int(csv_path.relative_to(root).parts[0].replace('noise_', ''))
            row['baseline'] = False
            row['noise_variant'] = f'noise_{row["noise_percent"]:03d}'
            row['csv_path'] = str(csv_path)
            rows.append(row)
    results = pd.DataFrame(rows)
    metadata_rows = []
    for dataset, path in metadata_paths.items():
        if path.exists():
            meta = pd.read_csv(path)
            if 'dataset' not in meta.columns:
                meta['dataset'] = dataset
            metadata_rows.append(meta)
    metadata = pd.concat(metadata_rows, ignore_index=True) if metadata_rows else pd.DataFrame()
    if results.empty:
        return results
    if not metadata.empty:
        results = results.merge(metadata, on=['dataset', 'noise_percent'], how='left', suffixes=('', '_metadata'))
    if 'graph_distance' not in results.columns:
        results['graph_distance'] = results['noise_percent'] / 100.0
    return results.sort_values(['dataset', 'noise_percent'])

def load_baseline_metrics(baseline_roots):
    rows = []
    for dataset, root in baseline_roots.items():
        if not root.exists():
            print(f'Baseline metrics root not found for {dataset}: {root}')
            continue
        candidates = []
        candidates.extend(sorted((root / 'last_metrics').glob('*.csv')))
        candidates.extend(sorted(root.glob('**/last_metrics/*.csv')))
        candidates.extend(
            csv_path for csv_path in sorted(root.glob('*.csv'))
            if not any(token in csv_path.name for token in SKIP_BASELINE_CSV_TOKENS)
        )
        candidates = _dedupe_paths(candidates)
        for csv_path in candidates:
            row = _read_first_metric_row(csv_path)
            if row is None:
                continue
            row['dataset'] = dataset
            row['noise_percent'] = 0
            row['graph_distance'] = 0.0
            row['baseline'] = True
            row['noise_variant'] = BASELINE_VARIANT
            row['csv_path'] = str(csv_path)
            rows.append(row)
            break
        else:
            print(f'No baseline metric CSV with expected metric columns found for {dataset}: {root}')
    return pd.DataFrame(rows)

def _attach_progressive_metadata(interventions, metadata_paths):
    metadata_rows = []
    for dataset, path in metadata_paths.items():
        if path.exists():
            meta = pd.read_csv(path)
            if 'dataset' not in meta.columns:
                meta['dataset'] = dataset
            metadata_rows.append(meta)
    metadata = pd.concat(metadata_rows, ignore_index=True) if metadata_rows else pd.DataFrame()
    if not metadata.empty:
        interventions = interventions.merge(metadata, on=['dataset', 'noise_percent'], how='left', suffixes=('', '_metadata'))
    if 'graph_distance' not in interventions.columns:
        interventions['graph_distance'] = interventions['noise_percent'] / 100.0
    return interventions

def load_interventions(dataset_roots, metadata_paths):
    rows = []
    for dataset, root in dataset_roots.items():
        for csv_path in sorted(root.glob('noise_*/lightning_logs/**/intervention_results.csv')):
            df = pd.read_csv(csv_path)
            if df.empty:
                continue
            df['dataset'] = dataset
            df['noise_percent'] = int(csv_path.relative_to(root).parts[0].replace('noise_', ''))
            df['baseline'] = False
            df['noise_variant'] = df['noise_percent'].map(lambda value: f'noise_{int(value):03d}')
            df['csv_path'] = str(csv_path)
            rows.append(df)
    interventions = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    if interventions.empty:
        return interventions
    return _attach_progressive_metadata(interventions, metadata_paths)

def load_baseline_interventions(baseline_roots):
    rows = []
    for dataset, root in baseline_roots.items():
        if not root.exists():
            print(f'Baseline intervention root not found for {dataset}: {root}')
            continue
        candidates = _dedupe_paths(sorted(root.glob('lightning_logs/**/intervention_results.csv')) + sorted(root.glob('**/intervention_results.csv')))
        if not candidates:
            print(f'No baseline intervention_results.csv found for {dataset}: {root}')
            continue
        csv_path = candidates[-1]
        df = pd.read_csv(csv_path)
        if df.empty:
            continue
        df['dataset'] = dataset
        df['noise_percent'] = 0
        df['graph_distance'] = 0.0
        df['baseline'] = True
        df['noise_variant'] = BASELINE_VARIANT
        df['csv_path'] = str(csv_path)
        rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

def intervention_accuracy_column(df):
    for col in ['test_task_accuracy', 'task_accuracy', 'accuracy', 'test_accuracy']:
        if col in df.columns:
            return col
    return None


## Saved Values Table

In [ ]:
progressive_results = load_last_metrics(dataset_roots, metadata_paths)
baseline_metrics = load_baseline_metrics(baseline_roots)
merged = pd.concat([baseline_metrics, progressive_results], ignore_index=True, sort=False)

value_cols = available(
    merged.columns,
    [
        'dataset', 'noise_variant', 'baseline', 'noise_percent', 'graph_distance',
        'num_flipped_positions', 'different_edges', 'added_edges', 'deleted_edges',
        'original_true_entries', 'perturbed_true_entries', 'perturbed_edges',
        'test_task_accuracy', 'test_concept_accuracy', 'test_dropout_task_accuracy',
        'CCI', 'PFI_concept_importance', 'PFI_side_importance',
        'csv_path',
    ],
)

if merged.empty:
    print('No progressive-noise or CUB CREAM_best baseline result CSVs found yet. Run the jobs/evaluation first.')
else:
    sort_cols = available(merged.columns, ['dataset', 'graph_distance', 'noise_percent', 'baseline'])
    summary_values = merged[value_cols].sort_values(sort_cols)
    summary_values.to_csv(summary_out, index=False)
    print(f'Saved summary values to {summary_out}')
    display(summary_values)


In [ ]:
merged

## Accuracy, CCI, and PFI Curves

In [ ]:
metrics_to_plot = [
    ('test_task_accuracy', 'Task accuracy'),
    ('test_concept_accuracy', 'Concept accuracy'),
    ('CCI', 'CCI'),
    ('PFI_concept_importance', 'PFI concept importance'),
    ('PFI_side_importance', 'PFI side importance'),
]

if merged.empty:
    print('No progressive-noise metrics available yet.')
else:
    x_col = 'graph_distance' if 'graph_distance' in merged.columns else 'noise_percent'
    for metric, label in metrics_to_plot:
        if metric not in merged.columns:
            print(f'Skipping {metric}: column not found in last_metrics.')
            continue
        for dataset, df in merged.groupby('dataset'):
            df = df.dropna(subset=[x_col, metric]).sort_values(x_col)
            if df.empty:
                continue
            fig, ax = plt.subplots(figsize=(7, 4))
            ax.plot(df[x_col], df[metric], marker='o', linewidth=2)
            ax.set_xlabel('Graph distance = different edge positions / total positions' if x_col == 'graph_distance' else 'Noise percent')
            ax.set_ylabel(label)
            ax.set_title(f'{dataset}: {label} vs progressive graph noise')
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()


## Combined Metric Curves

In [ ]:
if merged.empty:
    print('No progressive-noise metrics available yet.')
else:
    x_col = 'graph_distance' if 'graph_distance' in merged.columns else 'noise_percent'
    for dataset, df in merged.groupby('dataset'):
        df = df.sort_values(x_col)
        plot_cols = [metric for metric, _ in metrics_to_plot if metric in df.columns and df[metric].notna().any()]
        if not plot_cols:
            print(f'No plottable metric columns for {dataset}.')
            continue
        fig, ax = plt.subplots(figsize=(8, 4.5))
        for metric in plot_cols:
            ax.plot(df[x_col], df[metric], marker='o', linewidth=2, label=metric)
        ax.set_xlabel('Graph distance = different edge positions / total positions' if x_col == 'graph_distance' else 'Noise percent')
        ax.set_ylabel('Value')
        ax.set_title(f'{dataset}: progressive graph-noise metrics')
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()


## Intervention Values and Curves

In [ ]:
progressive_interventions = load_interventions(dataset_roots, metadata_paths)
baseline_interventions = load_baseline_interventions(baseline_roots)
interventions = pd.concat([baseline_interventions, progressive_interventions], ignore_index=True, sort=False)
acc_col = intervention_accuracy_column(interventions)

intervention_cols = available(
    interventions.columns,
    ['dataset', 'noise_variant', 'baseline', 'noise_percent', 'graph_distance', 'group_interventions', 'num_interventions', acc_col, 'csv_path'],
)

if interventions.empty:
    print('No intervention_results.csv files found yet. Progressive jobs write them under noise_*/lightning_logs/**/; CREAM_best should have one under lightning_logs/**/.')
else:
    sort_cols = available(interventions.columns, ['dataset', 'graph_distance', 'noise_percent', 'group_interventions', 'num_interventions'])
    intervention_values = interventions[intervention_cols].sort_values(sort_cols)
    intervention_values.to_csv(intervention_out, index=False)
    print(f'Saved intervention values to {intervention_out}')
    display(intervention_values)


In [ ]:
if interventions.empty or acc_col is None or 'num_interventions' not in interventions.columns:
    print('No plottable intervention curves yet.')
else:
    for dataset, dataset_df in interventions.groupby('dataset'):
        group_values = [None]
        if 'group_interventions' in dataset_df.columns:
            group_values = sorted(dataset_df['group_interventions'].dropna().unique())
        for group_value in group_values:
            df = dataset_df if group_value is None else dataset_df[dataset_df['group_interventions'] == group_value]
            if df.empty:
                continue
            hue_col = 'graph_distance' if 'graph_distance' in df.columns else 'noise_percent'
            fig, ax = plt.subplots(figsize=(8, 5))
            for hue_value, hue_df in df.groupby(hue_col):
                hue_df = hue_df.sort_values('num_interventions')
                ax.plot(hue_df['num_interventions'], hue_df[acc_col], marker='o', linewidth=2, label=str(hue_value))
            suffix = '' if group_value is None else f' | group_interventions={group_value}'
            ax.set_xlabel('Number of interventions')
            ax.set_ylabel(acc_col)
            ax.set_title(f'{dataset}: intervention curve under progressive graph noise{suffix}')
            ax.grid(True, alpha=0.3)
            ax.legend(title=hue_col, bbox_to_anchor=(1.02, 1), loc='upper left')
            plt.tight_layout()
            plt.show()
